#  Sellers - Bronze Ingestion

## Imports


In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_sellers"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist"
source_dataset = "sellers"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("seller_id", StringType(), True),
    StructField("seller_zip_code_prefix", StringType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

seller_id,seller_zip_code_prefix,seller_city,seller_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers


In [0]:
spark.table(target_table).count()

3095